<a href="https://colab.research.google.com/github/vijayalakshmish/Agent/blob/main/Agent2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Path to dataset files: /kaggle/input/brazilian-ecommerce


In [ ]:
"""
Olist ontology agent (Gemini) - an ontology agent on a real, multi-table dataset.

Dataset: "Brazilian E-Commerce Public Dataset by Olist" (Kaggle). Download it yourself,
check its license on the Kaggle page, and credit Olist. Do not commit the CSV files to GitHub.

Pipeline:  CSV files -> SQLite -> ontology (objects, links, actions) -> tools -> Gemini agent

Colab:
  !pip install -U google-genai pandas
  !unzip -o archive.zip -d olist            # the file you downloaded from Kaggle
  %run olist_agent.py --data olist          # first run builds olist_agent.db (about a minute)
  %run olist_agent.py                       # later runs reuse the database

Try it WITHOUT the real data (fake sample in the same format):
  %run olist_agent.py --make-sample sample
  %run olist_agent.py --data sample --db sample.db
Test everything with no AI and no key:
  %run olist_agent.py --selftest

Key: GEMINI_API_KEY environment variable or Colab secret (free key: aistudio.google.com/apikey).
"""
import argparse
import json
import math
import os
import random
import sqlite3
import sys
import tempfile
import time
from datetime import date, datetime, timedelta, timezone

DB_PATH = os.environ.get("OLIST_DB", "olist_agent.db")
AUDIT_PATH = os.environ.get("AUDIT_LOG", "audit_log.jsonl")
CONN = None
VERBOSE = False
LIMIT_MAX = 25  # tools never return more rows than this


class DataError(Exception):
    """Raised when the CSV files are missing or do not look like the Olist dataset."""


# ---------------------------------------------------------------- 1. DATA FILES
# name -> (csv file, columns we use). Extra columns in the files are ignored.
FILES = {
    "customers": ("olist_customers_dataset.csv",
                  ["customer_id", "customer_unique_id", "customer_city", "customer_state"]),
    "orders": ("olist_orders_dataset.csv",
               ["order_id", "customer_id", "order_status", "order_purchase_timestamp",
                "order_delivered_customer_date", "order_estimated_delivery_date"]),
    "order_items": ("olist_order_items_dataset.csv",
                    ["order_id", "order_item_id", "product_id", "seller_id", "price", "freight_value"]),
    "payments": ("olist_order_payments_dataset.csv",
                 ["order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value"]),
    "reviews": ("olist_order_reviews_dataset.csv", ["review_id", "order_id", "review_score"]),
    "products": ("olist_products_dataset.csv",
                 ["product_id", "product_category_name", "product_weight_g", "product_photos_qty"]),
    "sellers": ("olist_sellers_dataset.csv", ["seller_id", "seller_city", "seller_state"]),
}
TRANSLATION = ("product_category_name_translation.csv",
               ["product_category_name", "product_category_name_english"])  # optional


def find_data_folder(preferred=None):
    if preferred:
        if os.path.exists(os.path.join(preferred, FILES["orders"         ][0])):
            return preferred
        raise DataError(f"Could not find {FILES['orders'][0]} in '{preferred}'. "
                        "Unzip the Kaggle download into that folder.")
    for c in ("olist", "data", ".", "/content/olist", "/content", "/kaggle/input/brazilian-ecommerce"):
        if os.path.exists(os.path.join(c, FILES["orders"][0])):
            return c
    return None


def build_database(folder, db_path):
    """Read the CSV files, clean them, and write the data tables into SQLite."""
    import pandas as pd

    def read(name):
        fname, cols = FILES[name]
        path = os.path.join(folder, fname)
        if not os.path.exists(path):
            raise DataError(f"Missing file: {path}")
        head = [h.strip() for h in pd.read_csv(path, nrows=0).columns]
        missing = [c for c in cols if c not in head]
        if missing:
            raise DataError(f"{fname}: missing columns {missing}. Found: {head}")
        return pd.read_csv(path, usecols=cols)

    def ids(df, *cols):
        for c in cols:
            df[c] = df[c].astype(str).str.strip()
        return df

    def iso(series):  # one timestamp format everywhere, so text comparison = time comparison
        return pd.to_datetime(series, errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")

    customers = ids(read("customers"), "customer_id", "customer_unique_id").drop_duplicates("customer_id")
    orders = ids(read("orders"), "order_id", "customer_id").drop_duplicates("order_id")
    for c in ("order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"):
        orders[c] = iso(orders[c])
    items = ids(read("order_items"), "order_id", "product_id", "seller_id")
    items["order_item_id"] = pd.to_numeric(items["order_item_id"], errors="coerce").fillna(0).astype(int)
    items["item_key"] = items["order_id"] + "#" + items["order_item_id"].astype(str)
    for c in ("price", "freight_value"):
        items[c] = pd.to_numeric(items[c], errors="coerce")
    pays = ids(read("payments"), "order_id")
    pays["payment_key"] = pays["order_id"] + "#" + pays["payment_sequential"].astype(str)
    pays["payment_value"] = pd.to_numeric(pays["payment_value"], errors="coerce")
    reviews = ids(read("reviews"), "review_id", "order_id")
    reviews["review_score"] = pd.to_numeric(reviews["review_score"], errors="coerce")
    reviews = reviews.reset_index(drop=True)
    reviews["review_key"] = "RV" + (reviews.index + 1).astype(str)
    products = ids(read("products"), "product_id")
    trans_path = os.path.join(folder, TRANSLATION[0])
    if os.path.exists(trans_path):  # Portuguese -> English category names
        trans = pd.read_csv(trans_path, usecols=TRANSLATION[1])
        products = products.merge(trans, on="product_category_name", how="left")
        products["category"] = products["product_category_name_english"].fillna(products["product_category_name"])
        products = products.drop(columns=["product_category_name_english"])
    else:
        products["category"] = products["product_category_name"]
    products["category"] = products["category"].fillna("unknown")
    sellers = ids(read("sellers"), "seller_id").drop_duplicates("seller_id")

    conn = sqlite3.connect(db_path)
    try:
        tables = {"customers": customers, "orders": orders, "order_items": items, "payments": pays,
                  "reviews": reviews, "products": products, "sellers": sellers}
        for name, df in tables.items():
            df.to_sql(name, conn, if_exists="replace", index=False)
        for stmt in (
            "CREATE INDEX IF NOT EXISTS ix_orders_customer ON orders(customer_id)",
            "CREATE INDEX IF NOT EXISTS ix_orders_id ON orders(order_id)",
            "CREATE INDEX IF NOT EXISTS ix_items_order ON order_items(order_id)",
            "CREATE INDEX IF NOT EXISTS ix_items_seller ON order_items(seller_id)",
            "CREATE INDEX IF NOT EXISTS ix_items_product ON order_items(product_id)",
            "CREATE INDEX IF NOT EXISTS ix_pay_order ON payments(order_id)",
            "CREATE INDEX IF NOT EXISTS ix_rev_order ON reviews(order_id)",
            "CREATE INDEX IF NOT EXISTS ix_prod_id ON products(product_id)",
            "CREATE INDEX IF NOT EXISTS ix_sell_id ON sellers(seller_id)",
            "CREATE INDEX IF NOT EXISTS ix_cust_id ON customers(customer_id)",
            "DROP VIEW IF EXISTS orders_v",
            # Derived properties are COMPUTED here, not stored: they can never go stale.
            """CREATE VIEW orders_v AS SELECT order_id, customer_id, order_status,
                 order_purchase_timestamp, order_delivered_customer_date, order_estimated_delivery_date,
                 CASE WHEN order_delivered_customer_date IS NULL THEN NULL
                      WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 ELSE 0 END AS is_late,
                 CASE WHEN order_delivered_customer_date IS NULL THEN NULL
                      ELSE ROUND(julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp), 1)
                 END AS delivery_days
               FROM orders""",
        ):
            conn.execute(stmt)
        conn.commit()
    finally:
        conn.close()


def _has_data(path):
    if not os.path.exists(path):
        return False
    c = sqlite3.connect(path)
    try:
        return c.execute("SELECT 1 FROM sqlite_master WHERE name='orders_v'").fetchone() is not None
    finally:
        c.close()


def open_db(path, folder=None, rebuild=False):
    """Open (and build if needed) the database. Action tables are never dropped by a rebuild."""
    global CONN
    if rebuild or not _has_data(path):
        if not folder:
            raise DataError("No database yet and no CSV folder found. Download the Olist dataset from Kaggle, "
                            "unzip it, and run with --data FOLDER. (Or try --make-sample first.)")
        print(f"Building {path} from {folder} ...")
        build_database(folder, path)
    CONN = sqlite3.connect(path)
    CONN.row_factory = sqlite3.Row
    CONN.executescript("""
        CREATE TABLE IF NOT EXISTS order_flags (id INTEGER PRIMARY KEY, order_id TEXT, reason TEXT,
            status TEXT, created_at TEXT);
        CREATE TABLE IF NOT EXISTS refund_proposals (id INTEGER PRIMARY KEY, order_id TEXT, amount REAL,
            reason TEXT, status TEXT, created_at TEXT);
        CREATE TABLE IF NOT EXISTS seller_notes (id INTEGER PRIMARY KEY, seller_id TEXT, note TEXT,
            created_at TEXT);
    """)
    CONN.commit;
    return CONN


# ---------------------------------------------------------------- 2. ONTOLOGY
OBJECT_TYPES = {
    "Customer": {"table": "customers", "key": "customer_id",
                 "fields": ["customer_id", "customer_unique_id", "customer_city", "customer_state"]},
    "Order": {"table": "orders_v", "key": "order_id",
              "fields": ["order_id", "customer_id", "order_status", "order_purchase_timestamp",
                         "order_delivered_customer_date", "order_estimated_delivery_date",
                         "is_late", "delivery_days"]},
    "OrderItem": {"table": "order_items", "key": "item_key",
                  "fields": ["item_key", "order_id", "order_item_id", "product_id", "seller_id",
                             "price", "freight_value"]},
    "Product": {"table": "products", "key": "product_id",
                "fields": ["product_id", "category", "product_category_name",
                           "product_weight_g", "product_photos_qty"]},
    "Seller": {"table": "sellers", "key": "seller_id", "fields": ["seller_id", "seller_city", "seller_state"]},
    "Review": {"table": "reviews", "key": "review_key",
               "fields": ["review_key", "review_id", "order_id", "review_score"]},
    "Payment": {"table": "payments", "key": "payment_key",
                "fields": ["payment_key", "order_id", "payment_sequential", "payment_type",
                           "payment_installments", "payment_value"]},
}
# (from_type, link_name) -> (to_type, column on the from side, column on the to side, cardinality)
LINKS = {
    ("Order", "customer"): ("Customer", "customer_id", "customer_id", "one"),
    ("Order", "items"): ("OrderItem", "order_id", "order_id", "many"),
    ("Order", "payments"): ("Payment", "order_id", "order_id", "many"),
    ("Order", "reviews"): ("Review", "order_id", "order_id", "many"),
    ("Customer", "orders"): ("Order", "customer_id", "customer_id", "many"),
    ("OrderItem", "order"): ("Order", "order_id", "order_id", "one"),
    ("OrderItem", "product"): ("Product", "product_id", "product_id", "one"),
    ("OrderItem", "seller"): ("Seller", "seller_id", "seller_id", "one"),
    ("Product", "items"): ("OrderItem", "product_id", "product_id", "many"),
    ("Seller", "items"): ("OrderItem", "seller_id", "seller_id", "many"),
    ("Review", "order"): ("Order", "order_id", "order_id", "one"),
    ("Payment", "order"): ("Order", "order_id", "order_id", "one"),
}
OPS = {"=", "!=", "<", "<=", ">", ">=", "contains"}


# ---- Actions: validate (dry run, no writes) then apply (inside one transaction).
# Actions only INSERT into separate action tables. The original dataset is never edited.
def _need(cond, msg):
    if not cond:
        raise ValueError(msg)


def _text(value, lo, hi, label):
    s = str(value if value is not None else "").strip()
    _need(lo <= len(s) <= hi, f"{label} must be {lo} to {hi} characters")
    return s


def _exists(table, col, val):
    return CONN.execute(f"SELECT 1 FROM {table} WHERE {col} = ?", (val,)).fetchone() is not None


def _only(p, allowed):
    extra = set(p) - set(allowed)
    _need(not extra, f"Unexpected parameters: {sorted(extra)}. Allowed: {list(allowed)}")


def _validate_flag(p):
    _only(p, ("order_id", "reason"))
    oid = str(p.get("order_id", "")).strip()
    reason = _text(p.get("reason"), 3, 200, "reason")
    _need(_exists("orders", "order_id", oid), f"Unknown order '{oid}'")
    _need(CONN.execute("SELECT 1 FROM order_flags WHERE order_id=? AND status='open'", (oid,)).fetchone() is None,
          f"Order {oid} is already flagged")
    return {"order_id": oid, "reason": reason}


def _apply_flag(p, conn):
    conn.execute("INSERT INTO order_flags(order_id, reason, status, created_at) VALUES (?,?,?,?)",
                 (p["order_id"], p["reason"], "open", _now()))
    return f"Flagged order {p['order_id']} for review"


def _validate_refund(p):
    _only(p, ("order_id", "amount", "reason"))
    oid = str(p.get("order_id", "")).strip()
    reason = _text(p.get("reason"), 3, 200, "reason")
    try:
        amount = float(p.get("amount" ))
    except (TypeError, ValueError):
        raise ValueError("amount must be a number")
    _need(math.isfinite(amount) and amount > 0, "amount must be greater than 0")
    _need(_exists("orders", "order_id", oid), f"Unknown order '{oid}'")
    paid = CONN.execute("SELECT COALESCE(SUM(payment_value), 0) FROM payments WHERE order_id=?", (oid,)).fetchone()[0]
    _need(paid > 0, f"No payment recorded for order {oid}")
    already = CONN.execute("SELECT COALESCE(SUM(amount), 0) FROM refund_proposals "
                           "WHERE order_id=? AND status='proposed'", (oid,)).fetchone()[0]
    remaining = round(paid - already, 2)
    _need(round(amount, 2) <= remaining,
          f"Refund too large: order {oid} was paid {paid:.2f} BRL, {already:.2f} already proposed, "
          f"{remaining:.2f} BRL left")
    return {"order_id": oid, "amount": round(amount, 2), "reason": reason}


def _apply_refund(p, conn):
    conn.execute("INSERT INTO refund_proposals(order_id, amount, reason, status, created_at) VALUES (?,?,?,?,?)",
                 (p["order_id"], p["amount"], p["reason"], "proposed", _now()))
    return f"Recorded a refund PROPOSAL of {p['amount']:.2f} BRL for order {p['order_id']} (no money moved)"


def _validate_note(p):
    _only(p, ("seller_id", "note"))
    sid = str(p.get("seller_id", "")).strip()
    note = _text(p.get("note"), 3, 300, "note")
    _need(_exists("sellers", "seller_id", sid), f"Unknown seller '{sid}'")
    return {"seller_id": sid, "note": note}


def _apply_note(p, conn):
    conn.execute("INSERT INTO seller_notes(seller_id, note, created_at) VALUES (?,?,?)",
                 (p["seller_id"], p["note"], _now()))
    return f"Added a note to seller {p['seller_id']}"


ACTIONS = {
    "flag_order_for_review": {
        "description": "Mark an order for human review (once per open order).",
        "params": {"order_id": "string", "reason": "text, 3-200 characters"},
        "validate": _validate_flag, "apply": _apply_flag},
    "propose_refund": {
        "description": "Record a refund PROPOSAL in BRL. Cannot exceed what the customer paid minus earlier proposals.",
        "params": {"order_id": "string", "amount": "number in BRL", "reason": "text, 3-200 characters"},
        "validate": _validate_refund, "apply": _apply_refund},
    "add_seller_note": {
        "description": "Attach a note to a seller's record.",
        "params": {"seller_id": "string", "note": "text, 3-300 characters"},
        "validate": _validate_note, "apply": _apply_note},
}
# All three change data, so all need a human "yes" first.


def _now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def audit(event, **data):
    """Append one JSON line to the audit log. Logging must never crash the agent."""
    rec = {"time": _now(), "event": event, **data}
    try:
        with open(AUDIT_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, default=str) + "\n")
    except OSError:
        pass


def read_audit():
    try:
        with open(AUDIT_PATH, encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]
    except (OSError, json.JSONDecodeError):
        return []


def run_action(action, params_json, approve, who="human"):
    """Parse -> validate (dry run) -> ask a human -> apply in one transaction. Every step is audited."""
    spec = ACTIONS.get(action)
    if not spec:
        return {"error": f"Unknown action. Valid: {list(ACTIONS)}"}
    try:
        params = json.loads(params_json) if isinstance(params_json, str) else dict(params_json)
    except (json.JSONDecodeError, TypeError, ValueError):
        return {"error": "params_json must be a valid JSON object"}
    if not isinstance(params, dict):
        return {"error": "params_json must be a valid JSON object"}
    try:
        clean = spec["validate"](params)
    except ValueError as e:
        audit("action_blocked_by_rules", action=action, params=params, reason=str(e))
        return {"error": str(e)}
    audit("approval_requested", action=action, params=clean)
    if not approve(action, clean):
        audit("approval_decision", action=action, params=clean, decision="rejected", by=who)
        return {"status": "rejected_by_human"}
    audit("approval_decision", action=action, params=clean, decision="approved", by=who)
    try:
        with CONN:  # one transaction: commits on success, rolls back on any error
            result = spec["apply"](clean, CONN)
    except (sqlite3.Error, ValueError) as e:
        audit("action_failed", action=action, params=clean, error=str(e))
        return {"error": f"Action failed, nothing changed: {e}"}
    audit("action_executed", action=action, params=clean, approved_by=who, result=result)
    return {"status": "ok", "result": result}


def human_approval(name, params):
    print(f"\n[APPROVAL NEEDED] {name} {json.dumps(params)}")
    return input("Approve? (y/n): ").strip().lower() == "y"


# ---------------------------------------------------------------- 3. TOOLS
def _log(name, **kwargs):
    audit("tool_call", tool=name, args=kwargs)
    if VERBOSE:
        print(f"  tool> {name}({json.dumps(kwargs)})")


def _clamp(n, lo, hi):
    try:
        n = int(n)
    except (TypeError, ValueError):
        n = lo
    return max(lo, min(hi, n))


def _rows(sql, params=()):
    return [dict(r) for r in CONN.execute(sql, params)]


def search_objects(object_type: str, field: str = "", value: str = "", op: str = "=", limit: int = 10) -> dict:
    """Find objects of one type. Filter with field/op/value, e.g. object_type='Review', field='review_score',
    op='<=', value='2'. op is one of = != < <= > >= contains. Use value='null' to find missing values.
    Returns the total count and at most `limit` rows (max 25). Types: Customer, Order, OrderItem,
    Product, Seller, Review, Payment."""
    _log("search_objects", object_type=object_type, field=field, value=value, op=op, limit=limit)
    spec = OBJECT_TYPES.get(object_type)
    if not spec:
        return {"error": f"Unknown type. Valid: {list(OBJECT_TYPES)}"}
    if op not in OPS:
        return {"error": f"Unknown op. Valid: {sorted(OPS)}"}
    where, params = "", []
    if field:
        if field not in spec["fields"]:
            return {"error": f"Unknown field for {object_type}. Valid: {spec['fields']}"}
        if str(value).strip().lower() in ("null", "none") and op in ("=", "!="):
            where = f"WHERE {field} IS {'NOT ' if op == '!=' else ''}NULL"
        elif op == "contains":
            where, params = f"WHERE {field} LIKE ?", [f"%{value}%"]
        else:
            where, params = f"WHERE {field} {op} ? COLLATE NOCASE", [value]
    limit = _clamp(limit, 1, LIMIT_MAX)
    total = CONN.execute(f"SELECT COUNT(*) FROM {spec['table']} {where}", params).fetchone()[0]
    cols = ", ".join(spec["fields"])
    rows = _rows(f"SELECT {cols} FROM {spec['table']} {where} ORDER BY {spec['key']} LIMIT ?", [*params, limit])
    return {"count": total, "returned": len(rows), "results": rows}


def get_linked_objects(object_type: str, object_id: str, link: str, limit: int = 10) -> dict:
    """Follow a link from one object to related objects, e.g. object_type='Order', object_id='<order id>',
    link='items'. Returns the total count and at most `limit` rows (max 25)."""
    _log("get_linked_objects", object_type=object_type, object_id=object_id, link=link, limit=limit)
    spec = LINKS.get((object_type, link))
    if not spec:
        valid = [l for (t, l) in LINKS if t == object_type]
        return {"error": f"No link '{link}' on {object_type}. Valid: {valid}"}
    to_type, from_col, to_col, _ = spec
    src = OBJECT_TYPES[object_type]
    row = CONN.execute(f"SELECT {from_col} FROM {src['table']} WHERE {src['key']} = ?", (object_id,)).fetchone()
    if row is None:
        return {"error": f"{object_type} {object_id} not found"}
    dst = OBJECT_TYPES[to_type]
    total = CONN.execute(f"SELECT COUNT(*) FROM {dst['table']} WHERE {to_col} = ?", (row[0],)).fetchone()[0]
    cols = ", ".join(dst["fields"])
    rows = _rows(f"SELECT {cols} FROM {dst['table']} WHERE {to_col} = ? ORDER BY {dst['key']} LIMIT ?",
                 (row[0], _clamp(limit, 1, LIMIT_MAX)))
    return {"count": total, "returned": len(rows), "results": rows}


def delivery_stats(group_by: str = "", state: str = "", year: str = "", top: int = 10) -> dict:
    """Late-delivery statistics for DELIVERED orders (an order is late if delivered after the estimated date).
    group_by: '' (overall), 'state' (customer state, only states with 50+ delivered orders) or 'month'.
    Optional filters: state (like 'SP'), year (like '2017'). Returns delivered_orders, late_orders,
    late_pct and avg_delivery_days."""
    _log("delivery_stats", group_by=group_by, state=state, year=year, top=top)
    if group_by not in ("", "state", "month"):
        return {"error": "group_by must be '', 'state' or 'month'"}
    where, params = ["o.is_late IS NOT NULL"], []
    if state:
        where.append("c.customer_state = ? COLLATE NOCASE")
        params.append(state)
    if year:
        where.append("strftime('%Y', o.order_purchase_timestamp) = ?")
        params.append(str(year))
    group = {"": "", "state": "c.customer_state", "month": "strftime('%Y-%m', o.order_purchase_timestamp)"}[group_by]
    label = f"{group} AS grp," if group else ""
    sql = f"""SELECT {label} COUNT(*) AS delivered_orders, SUM(o.is_late) AS late_orders,
                     ROUND(100.0 * SUM(o.is_late) / COUNT(*), 1) AS late_pct,
                     ROUND(AVG(o.delivery_days), 1) AS avg_delivery_days
              FROM orders_v o JOIN customers c ON c.customer_id = o.customer_id
              WHERE {' AND '.join(where)}"""
    if group:
        having = "HAVING COUNT(*) >= 50" if group_by == "state" else ""
        order = "late_pct DESC" if group_by == "state" else "grp ASC"
        sql += f" GROUP BY grp {having} ORDER BY {order} LIMIT ?"
        params.append(_clamp(top, 1, 36))
    return {"results": _rows(sql, params)}


def seller_performance(min_orders: int = 30, sort_by: str = "late_rate", top: int = 10, state: str = "") -> dict:

    """Rank sellers. sort_by: 'late_rate' (highest first), 'avg_review' (lowest first) or 'orders' (most first).
    Only sellers with at least min_orders orders are listed (small samples are unreliable).
    late_pct is the share of the seller's delivered orders that arrived after the estimate;
    avg_review is the average review score (1-5)."""
    _log("seller_performance", min_orders=min_orders, sort_by=sort_by, top=top, state=state)
    order = {"late_rate": "late_pct DESC", "avg_review": "avg_review ASC", "orders": "orders DESC"}.get(sort_by)
    if not order:
        return {"error": "sort_by must be 'late_rate', 'avg_review' or 'orders'"}
    params = []
    st = ""
    if state:
        st = "WHERE s.seller_state = ? COLLATE NOCASE"
        params.append(state)
    sql = f"""SELECT s.seller_id, s.seller_state, COUNT(*) AS orders,
                     ROUND(100.0 * AVG(o.is_late), 1) AS late_pct, ROUND(AVG(r.score), 2) AS avg_review
              FROM (SELECT DISTINCT seller_id, order_id FROM order_items) so
              JOIN sellers s ON s.seller_id = so.seller_id
              JOIN orders_v o ON o.order_id = so.order_id
              LEFT JOIN (SELECT order_id, AVG(review_score) AS score FROM reviews GROUP BY order_id) r
                     ON r.order_id = so.order_id
              {st}
              GROUP BY s.seller_id HAVING COUNT(*) >= ?
              ORDER BY {order}, orders DESC LIMIT ?"""
    params += [_clamp(min_orders, 1, 100000), _clamp(top, 1, LIMIT_MAX)]
    return {"results": _rows(sql, params)}


def sales_summary(group_by: str = "category", top: int = 10, year: str = "") -> dict:
    """Revenue (sum of item prices, in BRL) and number of orders, excluding canceled and unavailable orders.
    group_by: 'category' (English product category), 'month' (purchase month) or 'state' (customer state).
    Optional year filter like '2017'. Categories and states are sorted by revenue; months by date."""
    _log("sales_summary", group_by=group_by, top=top, year=year)
    expr = {"category": "p.category", "month": "strftime('%Y-%m', o.order_purchase_timestamp)",
            "state": "c.customer_state"}.get(group_by)
    if not expr:
        return {"error": "group_by must be 'category', 'month' or 'state'"}
    where, params = ["o.order_status NOT IN ('canceled', 'unavailable')"], []
    if year:
        where.append("strftime('%Y', o.order_purchase_timestamp) = ?")
        params.append(str(year))
    order = "grp ASC" if group_by == "month" else "revenue DESC"
    sql = f"""SELECT {expr} AS grp, ROUND(SUM(i.price), 2) AS revenue_brl, COUNT(DISTINCT i.order_id) AS orders
              FROM order_items i
              JOIN orders_v o ON o.order_id = i.order_id
              JOIN products p ON p.product_id = i.product_id
              JOIN customers c ON c.customer_id = o.customer_id
              WHERE {' AND '.join(where)} GROUP BY grp ORDER BY {order.replace('revenue', 'revenue_brl')} LIMIT ?"""
    params.append(_clamp(top, 1, 36))
    return {"results": _rows(sql, params)}


def data_quality_report() -> dict:
    """Counts of known data problems: orders without items, delivered orders missing a delivery date,
    orders with several reviews, estimates before purchase, negative amounts, products with unknown
    category, items pointing to missing products or sellers, plus orders per status."""
    _log("data_quality_report")
    one = lambda sql: CONN.execute(sql).fetchone()[0]
    return {
        "orders": one("SELECT COUNT(*) FROM orders"),
        "orders_without_items": one("SELECT COUNT(*) FROM orders o WHERE NOT EXISTS "
                                    "(SELECT 1 FROM order_items i WHERE i.order_id = o.order_id)"),
        "delivered_orders_missing_delivery_date": one("SELECT COUNT(*) FROM orders WHERE order_status='delivered' "
                                                      "AND order_delivered_customer_date IS NULL"),
        "orders_with_multiple_reviews": one("SELECT COUNT(*) FROM (SELECT order_id FROM reviews "
                                            "GROUP BY order_id HAVING COUNT(*) > 1)"),
        "estimate_before_purchase": one("SELECT COUNT(*) FROM orders WHERE order_estimated_delivery_date "
                                        "< order_purchase_timestamp"),
        "items_with_negative_amount": one("SELECT COUNT(*) FROM order_items WHERE price < 0 OR freight_value < 0"),
        "products_with_unknown_category": one("SELECT COUNT(*) FROM products WHERE category = 'unknown'"),
        "items_with_missing_product": one("SELECT COUNT(*) FROM order_items i WHERE NOT EXISTS "
                                          "(SELECT 1 FROM products p WHERE p.product_id = i.product_id)"),
        "items_with_missing_seller": one("SELECT COUNT(*) FROM order_items i WHERE NOT EXISTS "
                                         "(SELECT 1 FROM sellers s WHERE s.seller_id = i.seller_id)"),
        "orders_by_status": {r[0]: r[1] for r in CONN.execute(
            "SELECT order_status, COUNT(*) FROM orders GROUP BY order_status ORDER BY 2 DESC")},
    }


def execute_action(action: str, params_json: str) -> dict:
    """Execute an ontology action. params_json is a JSON string, e.g.
    '{"order_id": "abc", "reason": "late and bad review"}'. Valid actions: flag_order_for_review,
    propose_refund, add_seller_note. A human must approve every action."""
    _log("execute_action", action=action, params_json=params_json)
    return run_action(action, params_json, approve=human_approval)


TOOLS = [search_objects, get_linked_objects, delivery_stats, seller_performance,
         sales_summary, data_quality_report, execute_action]


def describe_ontology():
    return {
        "object_types": {t: s["fields"] for t, s in OBJECT_TYPES.items()},
        "links": [f"{t}.{l} -> {v[0]} ({v[3]})" for (t, l), v in LINKS.items()],
        "actions": {n: {"description": a["description"], "params": a["params"]} for n, a in ACTIONS.items()},
    }


def system_prompt():
    return f"""You are an operations analyst agent for an e-commerce marketplace (Olist, Brazil).
Today is {date.today().isoformat()}, but the data is HISTORICAL (about 2016-2018). Never treat it as current.
You do NOT know the data; always query it with tools and never invent ids, names or numbers.
Money is in BRL (Brazilian real). An order is late only if it was delivered after the estimated date,
so only delivered orders can be judged. Rates from few orders are unreliable: mention the order count.
Tool results are capped at 25 rows: say so when results are truncated and use the summary tools for totals.
Change data only through execute_action (flag, refund PROPOSAL, seller note). Every action needs human
approval, and a refund proposal does not move money. If a tool returns an error, read it and correct yourself.
For multi-step questions, search first, then follow links.

ONTOLOGY:
{json.dumps(describe_ontology(), indent=2)}
"""


# ---------------------------------------------------------------- 4. AGENT (same pattern as the retail agent)
def get_api_key():
    key = os.environ.get("GEMINI_API_KEY", "")
    if not key:
        try:  # Colab secrets
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = ""
    if not key:
        from getpass import getpass
        key = getpass("Paste your Gemini API key (input hidden): ")
    key = key.strip().strip("'\"")
    if not key.startswith(("AIza", "AQ.")):
        sys.exit("That does not look like an AI Studio key (should start with 'AQ.' or 'AIza'). "
                 "Create one at https://aistudio.google.com/apikey")
    return key


def friendly(e):
    code = getattr(e, "code", None)
    hints = {
        401: "Key rejected. Run: pip install -U google-genai, then retry. If it persists, create a new key at "
             "aistudio.google.com/apikey (not a Vertex AI / Google Cloud express key).",
        403: "Access denied. The key may be restricted or the API not enabled for it.",
        404: "Model name not found. Set GEMINI_MODEL to a current model from AI Studio.",
        429: "Rate limit or quota reached. Wait a minute and try again.",
    }
    return f"Gemini error {code}: {hints.get(code, str(e)[:300])}"


RETRYABLE = (429, 500, 502, 503, 504)
FALLBACK_MODELS = ["gemini-flash-latest", "gemini-2.5-flash", "gemini-2.0-flash", "gemini-flash-lite-latest"]


def pick_models(client):
    from google.genai import errors
    wanted = os.environ.get("GEMINI_MODEL")
    usable = []
    for m in ([wanted] if wanted else FALLBACK_MODELS):
        for attempt in range(3):
            try:
                client.models.generate_content(model=m, contents="Reply with the word OK.")
                usable.append(m)
                break
            except errors.APIError as e:
                code = getattr(e, "code", None)
                if code == 404:
                    break
                if code in RETRYABLE:
                    if attempt == 2:
                        usable.append(m)
                        break
                    time.sleep(2 * (attempt + 1))
                    continue
                sys.exit(friendly(e))
    if not usable:
        sys.exit("No working Gemini model found. Set GEMINI_MODEL to a model listed in AI Studio.")
    return usable


def _ask(client, types, models, state, question):
    from google.genai import errors
    new_chat = lambda m: client.chats.create(model=m, config=types.GenerateContentConfig(
        system_instruction=system_prompt(), tools=TOOLS))
    if state.get("chat") is None:
        state["chat"], state["model"] = new_chat(models[0]), models[0]
    order = [state["model"]] + [m for m in models if m != state["model"]]
    last = None
    for m in order:
        if m != state["model"]:
            print(f"  (switching to model {m}; earlier chat memory is reset)")
            state["chat"], state["model"] = new_chat(m), m
        for attempt in range(4):
            try:
                return state["chat"].send_message(question).text or "(no text reply)"
            except errors.APIError as e:
                last = e
                if getattr(e, "code", None) not in RETRYABLE:
                    return friendly(e)
                wait = 3 * (attempt + 1)
                print(f"  (Google says the model is busy; retrying in {wait}s...)")
                time.sleep(wait)
    return friendly(last) + "\nAll models are busy right now. Wait a few minutes and try again."


def ask(client, types, models, state, question):
    audit("question", text=question)
    answer = _ask(client, types, models, state, question)
    audit("answer", model=state.get("model"), text=answer)
    return answer


def agent():
    global VERBOSE
    VERBOSE = True
    try:
        from google import genai
        from google.genai import types
    except ImportError:
        sys.exit("Install the SDK first:  pip install -U google-genai")
    os.environ.pop("GOOGLE_API_KEY", None)
    client = genai.Client(api_key=get_api_key())
    models = pick_models(client)
    state = {"chat": None, "model": None}
    print(f"Olist agent ready (models: {', '.join(models)}). Audit log: {AUDIT_PATH}")
    print("Try: 'Which sellers with at least 50 orders have the highest late-delivery rate?'  Type 'quit' to exit.\n")
    while True:
        try:
            q = input("> ").strip()
        except EOFError:
            break
        if q.lower() in ("quit", "exit", ""):
            break
        print("\n" + ask(client, types, models, state, q) + "\n")


# ---------------------------------------------------------------- 5. FAKE SAMPLE DATA (same format as Olist)
def make_sample(folder, n_orders=300, seed=7):
    """Write a small fake dataset with Olist's file and column names, including its known quirks
    (missing categories, categories without translation, undelivered orders, duplicate reviews)."""
    import csv
    rng = random.Random(seed)
    os.makedirs(folder, exist_ok=True)
    hexid = lambda: f"{rng.getrandbits(128):032x}"
    fmt = lambda d: d.strftime("%Y-%m-%d %H:%M:%S") if d else ""

    def write(name, header, rows):
        with open(os.path.join(folder, name), "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(header)
            w.writerows(rows)

    states = ["SP", "RJ", "MG", "RS", "BA"]
    cities = {"SP": "sao paulo", "RJ": "rio de janeiro", "MG": "belo horizonte", "RS": "porto alegre", "BA": "salvador"}
    cats = ["beleza_saude", "informatica_acessorios", "cama_mesa_banho", "esporte_lazer",
            "moveis_decoracao", "pet_shop"]
    english = {"beleza_saude": "health_beauty", "informatica_acessorios": "computers_accessories",
               "cama_mesa_banho": "bed_bath_table", "esporte_lazer": "sports_leisure",
               "moveis_decoracao": "furniture_decor"}  # pet_shop is deliberately missing
    sellers = [[hexid(), 13000 + i, cities[s], s] for i in range(12) for s in [rng.choice(states)]]
    products = [[hexid(), "" if i in (3, 17) else rng.choice(cats), 40, 200, 1,
                 rng.randint(100, 5000), 20, 10, 15] for i in range(40)]
    customers = [[hexid(), hexid(), 14000 + k, cities[s], s] for k in range(n_orders) for s in [rng.choice(states)]]

    orders, items, pays, reviews = [], [], [], []
    base = datetime(2017, 1, 1)
    for k in range(n_orders):
        oid, cust = hexid(), customers[k][0]
        purchase = base + timedelta(days=rng.randint(0, 560), hours=rng.randint(0, 23),
                                    minutes=rng.randint(0, 59), seconds=rng.randint(0, 59))
        r = rng.random()
        status = "delivered" if r < 0.88 else "shipped" if r < 0.93 else "canceled" if r < 0.97 else "unavailable"
        est = (purchase + timedelta(days=rng.randint(10, 20))).replace(hour=0, minute=0, second=0)
        carrier = purchase + timedelta(days=2) if status in ("delivered", "shipped") else None
        delivered = (purchase + timedelta(days=rng.randint(5, 26), hours=rng.randint(0, 23))
                     if status == "delivered" else None)
        orders.append([oid, cust, status, fmt(purchase), fmt(purchase + timedelta(hours=1)), fmt(carrier),
                       fmt(delivered), fmt(est)])
        total = 0.0
        if not (status in ("canceled", "unavailable") and rng.random() < 0.6):
            for n in range(rng.randint(1, 3)):
                price, freight = round(rng.uniform(10, 300), 2), round(rng.uniform(5, 40), 2)
                total += price + freight
                items.append([oid, n + 1, rng.choice(products)[0], rng.choice(sellers)[0],
                              fmt(purchase + timedelta(days=3)), price, freight])
        total = total or round(rng.uniform(20, 200), 2)
        parts = rng.randint(1, 2)
        first = round(total / parts, 2)
        for n in range(parts):
            value = first if n < parts - 1 else round(total - first * (parts - 1), 2)
            pays.append([oid, n + 1, rng.choice(["credit_card", "boleto", "voucher"]), rng.randint(1, 6), value])
        if status == "delivered":
            reviews.append([hexid(), oid, rng.choices([1, 2, 3, 4, 5], [8, 5, 10, 25, 52])[0], "", "",
                            fmt(delivered), fmt(delivered + timedelta(days=1))])
            if k < 60 and rng.random() < 0.15:  # a few orders get a second review
                reviews.append([hexid(), oid, rng.randint(1, 5), "", "", fmt(delivered), fmt(delivered + timedelta(days=3))])

    write("olist_customers_dataset.csv",
          ["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"], customers)
    write("olist_orders_dataset.csv",
          ["order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_approved_at",
           "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"], orders)
    write("olist_order_items_dataset.csv",
          ["order_id", "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value"], items)
    write("olist_order_payments_dataset.csv",
          ["order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value"], pays)
    write("olist_order_reviews_dataset.csv",
          ["review_id", "order_id", "review_score", "review_comment_title", "review_comment_message",
           "review_creation_date", "review_answer_timestamp"], reviews)
    write("olist_products_dataset.csv",
          ["product_id", "product_category_name", "product_name_lenght", "product_description_lenght",
           "product_photos_qty", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"],
          products)
    write("olist_sellers_dataset.csv",
          ["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"], sellers)
    write("product_category_name_translation.csv",
          ["product_category_name", "product_category_name_english"], [[k, v] for k, v in english.items()])


# ---------------------------------------------------------------- 6. SELF-TEST (no AI, no key)
def selftest():
    global AUDIT_PATH
    import pandas as pd
    tmp = tempfile.mkdtemp()
    AUDIT_PATH = os.path.join(tmp, "audit.jsonl")
    folder = os.path.join(tmp, "olist")
    make_sample(folder)
    results = []

    def check(label, cond):
        results.append(bool(cond))
        print(("PASS  " if cond else "FAIL  ") + label)

    # --- loader catches bad files
    bad = os.path.join(tmp, "bad")
    make_sample(bad)
    p = os.path.join(bad, FILES["orders"][0])
    text = open(p, encoding="utf-8").read().replace("order_status", "status", 1)
    open(p, "w", encoding="utf-8").write(text)
    try:
        build_database(bad, os.path.join(tmp, "bad.db"))
        check("missing column is caught", False)
    except DataError as e:
        check("missing column is caught", "missing columns" in str(e))
    os.remove(os.path.join(bad, FILES["sellers"][0]))
    try:
        build_database(bad, os.path.join(tmp, "bad2.db"))
        check("missing file is caught", False)
    except DataError as e:
        check("missing file is caught", "Missing file" in str(e) or "missing columns" in str(e))
    try:
        open_db(os.path.join(tmp, "none.db"), None)
        check("no data and no folder gives a clear error", False)
    except DataError as e:
        check("no data and no folder gives a clear error", "--data" in str(e))

    open_db(os.path.join(tmp, "t.db"), folder)
    rd = lambda n: pd.read_csv(os.path.join(folder, n))
    O, I, P, R, PR, S = (rd("olist_orders_dataset.csv"), rd("olist_order_items_dataset.csv"),
                         rd("olist_order_payments_dataset.csv"), rd("olist_order_reviews_dataset.csv"),
                         rd("olist_products_dataset.csv"), rd("olist_sellers_dataset.csv"))
    C = rd("olist_customers_dataset.csv")
    T = rd("product_category_name_translation.csv")

    # --- search and links
    check("order count matches the CSV", search_objects("Order")["count"] == len(O))
    check("filter is case-insensitive",
          search_objects("Order", "order_status", "DELIVERED")["count"] == (O.order_status == "delivered").sum())
    check("value 'null' finds missing values",
          search_objects("Order", "order_delivered_customer_date", "null")["count"] ==
          O.order_delivered_customer_date.isna().sum())
    check("op >= works", search_objects("Review", "review_score", "4", ">=")["count"] == (R.review_score >= 4).sum())
    check("op contains works", search_objects("Customer", "customer_city", "paulo", "contains")["count"] ==
          (C.customer_city.str.contains("paulo")).sum())
    check("limit is capped at 25", search_objects("OrderItem", limit=1000)["returned"] == LIMIT_MAX)
    check("unknown type/field/op/link give helpful errors",
          all("Valid" in r["error"] for r in (search_objects("Nope"), search_objects("Order", "zzz", "1"),
                                              search_objects("Order", "order_id", "1", "~"),
                                              get_linked_objects("Order", "x", "nope"))))
    oid = I.order_id.value_counts().index[0]
    check("Order -> items count matches", get_linked_objects("Order", oid, "items")["count"] == (I.order_id == oid).sum())
    item = get_linked_objects("Order", oid, "items")["results"][0]
    check("two hops: item -> seller -> items",
          get_linked_objects("Seller", get_linked_objects("OrderItem", item["item_key"], "seller")["results"][0]["seller_id"],
                             "items")["count"] >= 1)
    check("unknown object id is reported", "not found" in get_linked_objects("Order", "zzz", "items")["error"])

    # --- summaries against independent pandas calculations
    delivered = O[O.order_delivered_customer_date.notna()].copy()
    late = pd.to_datetime(delivered.order_delivered_customer_date) > pd.to_datetime(delivered.order_estimated_delivery_date)
    d = delivery_stats()["results"][0]
    check("delivery_stats overall matches pandas",
          d["delivered_orders"] == len(delivered) and d["late_orders" ] == late.sum() and
          d["late_pct"] == round(100 * late.mean(), 1))
    by_state = delivery_stats("state", top=36)["results"]
    check("delivery_stats by state only lists states with 50+ delivered orders",
          all(r["delivered_orders"] >= 50 for r in by_state))
    check("delivery_stats rejects a bad group_by", "error" in delivery_stats("zzz"))

    prod = PR.merge(T, on="product_category_name", how="left")
    prod["category"] = prod.product_category_name_english.fillna(prod.product_category_name).fillna("unknown")
    ok = O[~O.order_status.isin(["canceled", "unavailable"])]
    j = I.merge(ok, on="order_id").merge(prod[["product_id", "category"]], on="product_id")
    exp_cat = j.groupby("category").price.sum().round(2).to_dict()
    got_cat = {r["grp"]: r["revenue_brl"] for r in sales_summary("category", top=36)["results"]}
    check("sales_summary by category matches pandas", got_cat == exp_cat)
    check("untranslated category falls back to Portuguese, missing becomes 'unknown'",
          "pet_shop" in got_cat and "unknown" in got_cat)
    j["month"] = pd.to_datetime(j.order_purchase_timestamp).dt.strftime("%Y-%m")
    exp_m = j.groupby("month").price.sum().round(2).to_dict()
    got_m = {r["grp"]: r["revenue_brl" ] for r in sales_summary("month", top=36)["results"]}
    check("sales_summary by month matches pandas", {k: v for k, v in got_m.items()} == {k: exp_m[k] for k in got_m}
          and len(got_m) == min(36, len(exp_m)))

    so = I[["seller_id", "order_id"]].drop_duplicates().merge(delivered.assign(late=late.astype(int))[["order_id", "late"]],
                                                                 on="order_id", how="inner")
    rev = R.groupby("order_id").review_score.mean().rename("score").reset_index()
    allso = I[["seller_id", "order_id"]].drop_duplicates()
    allso = allso.merge(O[["order_id", "order_delivered_customer_date", "order_estimated_delivery_date"]], on="order_id")
    allso["late"] = (pd.to_datetime(allso.order_delivered_customer_date) >
                     pd.to_datetime(allso.order_estimated_delivery_date)).astype(float)
    allso.loc[allso.order_delivered_customer_date.isna(), "late"] = float("nan")
    allso = allso.merge(rev, on="order_id", how="left")
    g = allso.groupby("seller_id").agg(orders=("order_id", "count"), late=("late", "mean"), score=("score", "mean"))
    got = {r["seller_id"]: r for r in seller_performance(min_orders=1, top=25)["results"]}
    exp_ok = all(
        r["orders"] == g.loc[s, "orders"] and
        ((r["late_pct"] is None and pd.isna(g.loc[s, "late"])) or r["late_pct"] == round(100 * g.loc[s, "late"], 1)) and
        ((r["avg_review"] is None and pd.isna(g.loc[s, "score"])) or r["avg_review"] == round(g.loc[s, "score"], 2))
        for s, r in got.items())
    check("seller_performance matches pandas (orders, late %, review)", got and exp_ok)
    check("min_orders filters out small sellers",
          all(r["orders"] >= 30 for r in seller_performance(min_orders=30)["results"]))
    check("seller_performance rejects a bad sort", "error" in seller_performance(sort_by="zzz"))

    dq = data_quality_report()
    check("data_quality counts match pandas",
          dq["orders_without_items"] == (~O.order_id.isin(I.order_id)).sum() and
          dq["orders_with_multiple_reviews"] == (R.order_id.value_counts() > 1).sum() and
          dq["products_with_unknown_category"] == PR.product_category_name.isna().sum() and
          dq["orders"] == len(O))
    check("data_quality finds the planted problems",
          dq["orders_with_multiple_reviews"] > 0 and dq["orders_without_items"] > 0)

    # --- actions: validated, approved, transactional, original data untouched
    yes, no = (lambda *a: True), (lambda *a: False)
    before = CONN.execute("SELECT COUNT(*), SUM(price) FROM order_items").fetchone()[:]
    o1 = str(O.order_id.iloc[0])
    f = json.dumps({"order_id": o1, "reason": "late and bad review"})
    check("rejected action changes nothing",
          run_action("flag_order_for_review", f, no)["status"] == "rejected_by_human" and
          CONN.execute("SELECT COUNT(*) FROM order_flags").fetchone()[0] == 0)
    check("approved flag is recorded", run_action("flag_order_for_review", f, yes)["status"] == "ok" and
          CONN.execute("SELECT COUNT(*) FROM order_flags").fetchone()[0] == 1)
    check("same order cannot be flagged twice (idempotent)", "already flagged" in run_action("flag_order_for_review", f, yes)["error"])
    check("unknown order is refused", "Unknown order" in run_action(
        "flag_order_for_review", '{"order_id":"nope","reason":"abc"}', yes)["error"])
    check("short reason is refused", "reason" in run_action(
        "flag_order_for_review", json.dumps({"order_id": str(O.order_id.iloc[1]), "reason": "x"}), yes)["error"])
    check("unexpected parameter is refused", "Unexpected" in run_action(
        "flag_order_for_review", json.dumps({"order_id": o1, "reason": "abcd", "admin": True}), yes)["error"])
    paid = float(P[P.order_id == o1].payment_value.sum())
    r1 = run_action("propose_refund", json.dumps({"order_id": o1, "amount": round(paid / 2, 2), "reason": "late delivery"}), yes)
    check("refund within the paid amount is recorded as a proposal", r1.get("status") == "ok" and "no money moved" in r1["result"])
    check("refund above the remaining amount is blocked",
          "Refund too large" in run_action("propose_refund", json.dumps(
              {"order_id": o1, "amount": round(paid, 2), "reason": "too much"}), yes)["error"])
    check("negative or text amounts are blocked",
          all("error" in run_action("propose_refund", json.dumps({"order_id": o1, "amount": a, "reason": "abc"}), yes)
              for a in (-5, "lots", 0)))
    check("seller note works and unknown seller is refused",
          run_action("add_seller_note", json.dumps({"seller_id": str(S.seller_id.iloc[0]), "note": "slow shipping"}), yes)["status"] == "ok"
          and "Unknown seller" in run_action("add_seller_note", '{"seller_id":"zz","note":"abc"}', yes)["error"])
    check("bad JSON and unknown action are handled",
          "error" in run_action("flag_order_for_review", "not json", yes) and
          "error" in run_action("delete_everything", "{}", yes))
    after = CONN.execute("SELECT COUNT(*), SUM(price) FROM order_items").fetchone()[:]
    check("original dataset is untouched by actions", before == after)

    # --- audit log
    events = {r["event"] for r in read_audit()}
    for needed in ("tool_call", "approval_requested", "approval_decision", "action_executed", "action_blocked_by_rules"):
        check(f"audit log contains '{needed}'", needed in events)

    # --- rebuild keeps action tables
    build_database(folder, os.path.join(tmp, "t.db"))
    CONN.close()
    open_db(os.path.join(tmp, "t.db"), folder)
    check("rebuilding the data keeps flags and proposals",
          CONN.execute("SELECT COUNT(*) FROM order_flags").fetchone()[0] == 1)

    # --- Gemini can build a schema from every tool (only if the SDK is installed)
    try:
        from google.genai import types
        for t in TOOLS:
            types.FunctionDeclaration.from_callable_with_api_option(callable=t, api_option="GEMINI_API")
        check("Gemini accepts all tool schemas", True)
    except ImportError:
        print("SKIP  Gemini schema check (google-genai not installed)")
    except Exception as e:
        check(f"Gemini accepts all tool schemas ({e})", False)

    print(f"\n{sum(results)}/{len(results)} checks passed")
    return all(results)


# ---------------------------------------------------------------- 7. ENTRY POINT
def main():
    global DB_PATH
    ap = argparse.ArgumentParser(description="Olist ontology agent")
    ap.add_argument("--selftest", action="store_true")
    ap.add_argument("--make-sample", metavar="DIR", help="write a small fake dataset in Olist's format")
    ap.add_argument("--data", metavar="DIR", help="folder with the Olist CSV files")
    ap.add_argument("--db", help="SQLite file to use (default olist_agent.db)")
    ap.add_argument("--rebuild", action="store_true", help="re-read the CSV files (keeps flags and proposals)")
    args, _ = ap.parse_known_args()

    if args.selftest:
        sys.exit(0 if selftest() else 1)
    if args.make_sample:
        make_sample(args.make_sample)
        print(f"Fake sample written to {args.make_sample}. Next:\n"
              f"  python olist_agent.py --data {args.make_sample} --db sample.db")
        return
    if args.db:
        DB_PATH = args.db
    try:
        folder = find_data_folder(args.data) if (args.data or not _has_data(DB_PATH)) else None
        open_db(DB_PATH, folder, rebuild=args.rebuild)
    except DataError as e:
        sys.exit(f"Data problem: {e}")
    n = CONN.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
    print(f"Database ready: {DB_PATH} ({n:,} orders)")
    agent()


if __name__ == "__main__":
    main()


Database ready: olist_agent.db (99,441 orders)
Olist agent ready (models: gemini-flash-latest, gemini-flash-lite-latest). Audit log: audit_log.jsonl
Try: 'Which sellers with at least 50 orders have the highest late-delivery rate?'  Type 'quit' to exit.

> Which sellers with at least 50 orders have the highest late-delivery rate?
  tool> seller_performance({"min_orders": 50, "sort_by": "late_rate", "top": 10, "state": ""})

Based on historical data (2016–2018), here are the top 10 sellers with at least 50 orders ranked by the highest late-delivery rate (share of delivered orders arriving after the estimated delivery date):

| Rank | Seller ID | State | Total Orders | Late Delivery Rate (%) | Avg Review Score |
|:---:|:---|:---:|:---:|:---:|:---:|
| 1 | `54965bbe3e4f07ae045b90b0b8541f52` | PR | 78 | **30.1%** | 3.00 |
| 2 | `2a1348e9addc1af5aaa619b1a3679d6b` | MG | 52 | **27.1%** | 3.12 |
| 3 | `a49928bcdf77c55c6d6e05e09a9b4ca5` | SP | 98 | **26.0%** | 3.01 |
| 4 | `beadbee30901a7f61d031